# DebugZero OpenEnv GRPO Training Notebook

This Colab notebook installs DebugZero from GitHub, starts the OpenEnv server, connects through the packaged OpenEnv client, and trains a small code model with TRL GRPO/Unsloth. The reward function calls `env.reset()` and `env.step()` on the live environment for every completion, so training is tied to the actual environment instead of a static dataset or local file import.

## What this notebook proves

- Installs the submitted environment directly from GitHub.
- Uses the standard OpenEnv client/server boundary.
- Runs a smoke test against the live environment.
- Trains with GRPO using environment rewards from real rollouts.
- Saves loss/reward plots that can be committed into the README.

In [ ]:
# Change these only if your final submission branch or Space URL differs.
REPO_URL = "https://github.com/Ray-0906/DebugZero.git"
BRANCH = "main"

# Leave blank to launch the environment locally from the GitHub package.
# Set this to your Hugging Face Space URL to train against the deployed Space instead.
REMOTE_OPENENV_URL = ""

MODEL_ID = "unsloth/Qwen2.5-Coder-3B-Instruct"
OUTPUT_DIR = "debugzero-openenv-grpo"
MAX_STEPS = 30
NUM_GENERATIONS = 2
RUN_TRAINING = True

In [ ]:
import subprocess
import sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Install runtime/training dependencies explicitly, then install the environment package
# from GitHub without letting the repo's experimental training pins override Colab.
pip_install(
    "openenv-core[core]>=0.2.1",
    "datasets",
    "trl",
    "transformers",
    "accelerate",
    "peft",
    "bitsandbytes",
    "matplotlib",
    "pandas",
    "thefuzz[speedup]",
    "uvicorn[standard]",
    "requests",
)

try:
    pip_install("unsloth")
except Exception as exc:
    print(f"Unsloth install failed, continuing with native TRL fallback: {exc}")

pip_install("--no-deps", f"git+{REPO_URL}@{BRANCH}")

In [ ]:
import atexit
import os
import subprocess
import sys
import time

import requests

if REMOTE_OPENENV_URL:
    BASE_URL = REMOTE_OPENENV_URL.rstrip("/")
    server_process = None
else:
    BASE_URL = "https://huggingface.co/spaces/The-Fool-09/debugZero"
    server_process = subprocess.Popen(
        [sys.executable, "-m", "debugZero.server.app", "--host", "127.0.0.1", "--port", "8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    atexit.register(lambda: server_process and server_process.poll() is None and server_process.terminate())

def wait_for_openenv(base_url, timeout_s=90):
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url}/schema", timeout=5)
            if response.status_code == 200:
                return response.json()
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    if server_process and server_process.stdout:
        print(server_process.stdout.read())
    raise RuntimeError(f"OpenEnv server did not become ready: {last_error}")

schema = wait_for_openenv(BASE_URL)
print("Connected to OpenEnv:", BASE_URL)
schema

In [ ]:
from debugZero.client import DebugzeroEnv
from debugZero.models import DebugzeroAction

def observation(result):
    return getattr(result, "observation", result)

with DebugzeroEnv(base_url=BASE_URL) as env:
    reset_obs = observation(env.reset())
    print("Initial role:", reset_obs.role_next)
    print(reset_obs.current_code[:300])

    buggy_code = reset_obs.current_code.replace("distance < threshold", "distance <= threshold")
    prop_obs = observation(env.step(DebugzeroAction(role="proposer", code=buggy_code)))
    print("After proposer:", {"role_next": prop_obs.role_next, "tests_passed": prop_obs.tests_passed, "syntax_error": prop_obs.syntax_error})

    solve_obs = observation(env.step(DebugzeroAction(role="solver", code=reset_obs.current_code)))
    print("After solver:", {"role_next": solve_obs.role_next, "tests_passed": solve_obs.tests_passed, "syntax_error": solve_obs.syntax_error})

In [ ]:
import re
from datasets import Dataset

PROPOSER_PROMPT = """You are the Proposer in DebugZero.
Inject one realistic, syntax-valid bug into the clean Python function.
The mutated function should run but fail at least one hidden test.
Return only the complete modified Python code inside a python code fence.

Clean function:
```python
{code}
```
"""

SOLVER_PROMPT = """You are the Solver in DebugZero.
Repair the buggy Python function so it passes the environment tests.
Return only the complete repaired Python code inside a python code fence.

Buggy function:
```python
{code}
```
"""

def extract_code(text):
    if isinstance(text, list):
        text = text[0].get("content", "") if text else ""
    match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return (match.group(1) if match else text).strip()

def deterministic_bug(clean_code):
    replacements = [
        ("distance < threshold", "distance <= threshold"),
        ("return True", "return False"),
        ("idx != idx2", "idx == idx2"),
    ]
    for old, new in replacements:
        if old in clean_code:
            return clean_code.replace(old, new, 1)
    return clean_code + "\n# BUG: intentionally left for solver\n"

def build_openenv_dataset(num_rounds=8):
    rows = []
    with DebugzeroEnv(base_url=BASE_URL) as env:
        for episode in range(num_rounds):
            clean_obs = observation(env.reset())
            clean_code = clean_obs.current_code
            buggy_code = deterministic_bug(clean_code)

            rows.append({
                "prompt": PROPOSER_PROMPT.format(code=clean_code),
                "role": "proposer",
                "clean_code": clean_code,
                "buggy_code": "",
                "episode": episode,
            })

            rows.append({
                "prompt": SOLVER_PROMPT.format(code=buggy_code),
                "role": "solver",
                "clean_code": clean_code,
                "buggy_code": buggy_code,
                "episode": episode,
            })
    return Dataset.from_list(rows)

train_dataset = build_openenv_dataset(num_rounds=8)
train_dataset

In [ ]:
def proposer_reward(obs, submitted_code, clean_code):
    if obs.syntax_error:
        return -1.0
    if submitted_code.strip() == clean_code.strip():
        return -0.5
    if obs.tests_passed:
        return 0.0
    return 1.0

def solver_reward(obs, submitted_code, clean_code):
    if obs.syntax_error:
        return -1.0
    if submitted_code.strip() == clean_code.strip() and obs.tests_passed:
        return 1.25
    return 1.0 if obs.tests_passed else 0.0

def openenv_reward(completions, role=None, clean_code=None, buggy_code=None, **kwargs):
    rewards = []
    role = role or kwargs.get("roles")
    clean_code = clean_code or kwargs.get("clean_codes")
    buggy_code = buggy_code or kwargs.get("buggy_codes")

    with DebugzeroEnv(base_url=BASE_URL) as env:
        for completion, sample_role, clean, bug in zip(completions, role, clean_code, buggy_code):
            code = extract_code(completion)
            env.reset()
            if sample_role == "proposer":
                obs = observation(env.step(DebugzeroAction(role="proposer", code=code)))
                rewards.append(proposer_reward(obs, code, clean))
            elif sample_role == "solver":
                env.step(DebugzeroAction(role="proposer", code=bug))
                obs = observation(env.step(DebugzeroAction(role="solver", code=code)))
                rewards.append(solver_reward(obs, code, clean))
            else:
                rewards.append(0.0)
    return rewards

# Quick reward sanity checks against the live environment.
example = train_dataset[1]
print(openenv_reward(
    [f"```python\n{example['clean_code']}\n```"],
    role=[example["role"]],
    clean_code=[example["clean_code"]],
    buggy_code=[example["buggy_code"]],
))

In [ ]:
import torch

try:
    from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported
    PatchFastRL("GRPO", FastLanguageModel)
    HAS_UNSLOTH = True
except Exception as exc:
    print("Using native Transformers/TRL fallback:", exc)
    HAS_UNSLOTH = False
    is_bfloat16_supported = lambda: False

from trl import GRPOConfig, GRPOTrainer

if HAS_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        fast_inference=False,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    fallback_model = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(fallback_model, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        fallback_model,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

In [ ]:
def generate_completion(prompt, max_new_tokens=384):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def evaluate_policy(dataset, n=4):
    rows = [dataset[i] for i in range(min(n, len(dataset)))]
    completions = [generate_completion(row["prompt"]) for row in rows]
    rewards = openenv_reward(
        completions,
        role=[row["role"] for row in rows],
        clean_code=[row["clean_code"] for row in rows],
        buggy_code=[row["buggy_code"] for row in rows],
    )
    return rewards, completions

baseline_rewards, baseline_completions = evaluate_policy(train_dataset, n=4)
print("Baseline rewards:", baseline_rewards, "mean=", sum(baseline_rewards) / len(baseline_rewards))

In [ ]:
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=1024,
    max_completion_length=384,
    logging_steps=1,
    save_steps=MAX_STEPS,
    report_to="none",
    bf16=bool(torch.cuda.is_available() and is_bfloat16_supported()),
    fp16=bool(torch.cuda.is_available() and not is_bfloat16_supported()),
)

trainer_kwargs = dict(
    model=model,
    reward_funcs=[openenv_reward],
    args=training_args,
    train_dataset=train_dataset,
)

try:
    trainer = GRPOTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = GRPOTrainer(tokenizer=tokenizer, **trainer_kwargs)

if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(OUTPUT_DIR)
else:
    train_result = None
    print("RUN_TRAINING=False, trainer configured but not executed.")

In [ ]:
trained_rewards, trained_completions = evaluate_policy(train_dataset, n=4)
print("Baseline rewards:", baseline_rewards, "mean=", sum(baseline_rewards) / len(baseline_rewards))
print("Trained rewards:", trained_rewards, "mean=", sum(trained_rewards) / len(trained_rewards))

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("results", exist_ok=True)
history = pd.DataFrame(trainer.state.log_history)
history.to_csv("results/training_log.csv", index=False)

reward_cols = [col for col in history.columns if "reward" in col.lower()]
loss_cols = [col for col in history.columns if "loss" in col.lower()]

if reward_cols:
    ax = history.plot(x="step", y=reward_cols, marker="o", figsize=(8, 4))
    ax.set_xlabel("training step")
    ax.set_ylabel("reward")
    ax.set_title("DebugZero OpenEnv reward during GRPO")
    plt.tight_layout()
    plt.savefig("results/reward_curve.png", dpi=160)
    plt.show()

if loss_cols:
    ax = history.plot(x="step", y=loss_cols, marker="o", figsize=(8, 4))
    ax.set_xlabel("training step")
    ax.set_ylabel("loss")
    ax.set_title("DebugZero GRPO loss")
    plt.tight_layout()
    plt.savefig("results/loss_curve.png", dpi=160)
    plt.show()

comparison = pd.DataFrame({
    "phase": ["baseline", "trained"],
    "mean_reward": [sum(baseline_rewards) / len(baseline_rewards), sum(trained_rewards) / len(trained_rewards)],
})
ax = comparison.plot.bar(x="phase", y="mean_reward", legend=False, figsize=(5, 4))
ax.set_xlabel("policy")
ax.set_ylabel("mean live OpenEnv reward")
ax.set_title("Before vs after training")
plt.tight_layout()
plt.savefig("results/baseline_vs_trained_reward.png", dpi=160)
plt.show()

comparison

## Submission note

After running the notebook, commit `results/reward_curve.png`, `results/loss_curve.png`, and `results/baseline_vs_trained_reward.png` or upload them to the README/blog. Judges should see the GitHub install cell, the OpenEnv smoke test, and the before/after reward comparison.